# Text Classification Using TFIDF and XGBoost

Robust TF-IDF + XGBoost pipeline for large, imbalanced corpora

In [1]:
# yelp_tfidf_xgb_gensim.py
# Multi-class (5 stars) imbalanced classification on Yelp Review Full
# - Light gensim cleaning (no stemming, no stopword removal)
# - Negation-safe stopwords for TF-IDF (keeps "no/not/nor/never")
# - TF-IDF (float32) + XGBoost with early stopping & class-balanced weights
# - Star names in classification report + normalized confusion matrix
# - GPU-aware via env var: export XGB_USE_GPU=1

import os, numpy as np, pandas as pd, scipy.sparse as sp
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction import text as sktext
from xgboost import XGBClassifier

In [2]:
!pip install gensim

In [3]:
# ---- gensim light cleaning (no stemming, no stopword removal) ----
from gensim.parsing import preprocessing as gpp

RANDOM_STATE = 42

# 0) Gensim filters: light & safe for sentiment
G_FILTERS = [
    gpp.strip_tags,
    gpp.strip_punctuation,
    gpp.strip_numeric,
    gpp.strip_multiple_whitespaces,
    gpp.strip_short,          # drops very short tokens (len<3)
    # DO NOT include gpp.remove_stopwords here (we'll handle stopwords in TF-IDF with a custom list)
    # DO NOT include stemming (hurts interpretability for star ratings)
]

def clean_with_gensim(texts):
    # preprocess_string -> list of tokens; join back to space-separated string
    return [" ".join(gpp.preprocess_string(t if isinstance(t, str) else "", filters=G_FILTERS)) for t in texts]

In [4]:
# 1) Load Yelp/yelp_review_full (labels 0..4 → 1..5 stars conceptually)
ds = load_dataset("Yelp/yelp_review_full")
X_all = ds["train"]["text"]
y_all = np.array(ds["train"]["label"], dtype=np.int64)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

yelp_review_full/train-00000-of-00001.pa(…):   0%|          | 0.00/299M [00:00<?, ?B/s]

yelp_review_full/test-00000-of-00001.par(…):   0%|          | 0.00/23.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/650000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [5]:
# Optional peek at class counts (0..4)
uniq, cnts = np.unique(y_all, return_counts=True)
print("Raw train distribution {label: count}:", dict(zip(uniq, cnts)))


Raw train distribution {label: count}: {0: 130000, 1: 130000, 2: 130000, 3: 130000, 4: 130000}


In [6]:
# 2) Train/Val split from official train; use official test for final evaluation
X_train, X_val, y_train, y_val = train_test_split(
    list(X_all), y_all, test_size=0.10, stratify=y_all, random_state=RANDOM_STATE
)
X_test = ds["test"]["text"]
y_test = np.array(ds["test"]["label"], dtype=np.int64)

print("Sizes -> Train:", len(X_train), " Val:", len(X_val), " Test:", len(X_test))

Sizes -> Train: 585000  Val: 65000  Test: 50000


Wrong key type: 132038 of type '<class 'numpy.int64'>' Expected one of int, slice, range, str or iterable

The error occurs because X_all is a Dataset object from the datasets library, which doesn't support the type of indexing that train_test_split is trying to use. I'll convert X_all to a list before splitting the data.

In [7]:
# 3) Clean with gensim (consistent across splits)
print("Cleaning text with gensim (light)…")
X_train_clean = clean_with_gensim(X_train)
X_val_clean   = clean_with_gensim(X_val)
X_test_clean  = clean_with_gensim(X_test)

Cleaning text with gensim (light)…


In [8]:
# 4) Negation-safe stopwords for TF-IDF
BASE_SW = sktext.ENGLISH_STOP_WORDS
CUSTOM_STOPWORDS = set(BASE_SW) - {"no", "not", "nor", "never"}

tfidf = TfidfVectorizer(
    stop_words=list(CUSTOM_STOPWORDS), # Convert set to list
    lowercase=True,
    dtype=np.float32,        # memory win
    max_features=50_000,     # adjust to your RAM/VRAM (50k–300k typical)
    ngram_range=(1, 2),      # unigrams + bigrams capture phrases like "not good"
    min_df=5,                # prune very rare tokens for large corpora
    max_df=0.95,             # drop overly common tokens
    sublinear_tf=True,       # log-scale TF
    norm="l2",
)

print("Fitting TF-IDF…")
Xtr = tfidf.fit_transform(X_train_clean)
Xva = tfidf.transform(X_val_clean)
Xte = tfidf.transform(X_test_clean)
assert sp.issparse(Xtr) and Xtr.dtype == np.float32
print("Completed Fitting TF-IDF")

Fitting TF-IDF…
Completed Fitting TF-IDF


In [9]:
# 5) Class-balanced per-example weights (multi-class)
classes = np.arange(5)  # labels 0..4
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight_map = {c: float(weights[i]) for i, c in enumerate(classes)}
sample_weights = np.vectorize(class_weight_map.get)(y_train).astype(np.float32)
print("Class weights (0..4):", class_weight_map)


Class weights (0..4): {0: 1.0, 1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0}


In [10]:
tree_method = "gpu_hist" if os.environ.get("XGB_USE_GPU", "0") == "1" else "hist"
print("Using tree method:", tree_method)

Using tree method: hist


To enable the GPU for XGBoost, you need to set the `XGB_USE_GPU` environment variable to "1".

In [11]:
%env XGB_USE_GPU=1

env: XGB_USE_GPU=1


In [12]:
tree_method = "gpu_hist" if os.environ.get("XGB_USE_GPU", "0") == "1" else "hist"
print("Using tree method:", tree_method)

Using tree method: gpu_hist


In [16]:
# 6) XGBoost config (CPU/GPU-aware) + early stopping
# tree_method = "gpu_hist" if os.environ.get("XGB_USE_GPU", "0") == "1" else "hist"
# if os.environ.get("XGB_USE_GPU", "0") == "1" else "cpu",
clf = XGBClassifier(
    objective="multi:softprob",
    num_class=len(classes),
    n_estimators=2000,       # high cap; ES will stop earlier
    learning_rate=0.05,
    max_depth=6,             # try 6–10 for text
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    reg_alpha=0.0,
    tree_method="hist", # "hist" or "gpu_hist"
    device="cuda",
    eval_metric="mlogloss",
    n_jobs=-1,
    random_state=RANDOM_STATE,
    early_stopping_rounds=50,
)

print(f"Training XGBoost (tree_method={tree_method}) with early stopping…")
clf.fit(
    Xtr, y_train,
    sample_weight=sample_weights,
    eval_set=[(Xva, y_val)],
    verbose=100
)

Training XGBoost (tree_method=gpu_hist) with early stopping…
[0]	validation_0-mlogloss:13.08158
[49]	validation_0-mlogloss:29.46259


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device='cuda', early_stopping_rounds=50,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=2000, n_jobs=-1, num_class=5, ...)

In [17]:
print(y_train.min(), y_train.max())


0 4


In [18]:
num_class = len(np.unique(y_train))
print(num_class)

5


In [ ]:
# 7) Evaluation with human-readable star names
star_names = [
    "1 star (very negative)",
    "2 stars (negative)",
    "3 stars (neutral)",
    "4 stars (positive)",
    "5 stars (very positive)"
]

y_pred = clf.predict(Xte)

print("\nAccuracy:", f"{accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report (with star names):")
print(classification_report(y_test, y_pred, target_names=star_names, digits=4))

cm = confusion_matrix(y_test, y_pred, normalize="true")
cm_df = pd.DataFrame(cm, index=star_names, columns=star_names)
print("\nNormalized Confusion Matrix (rows = true classes):")
print(cm_df.round(3))

In [ ]:
# 8) (Optional) Persist components
try:
    import joblib
    joblib.dump(tfidf, "yelp_tfidf_vectorizer.pkl", compress=3)
    clf.save_model("yelp_xgb_model.json")
    print("\nSaved: yelp_tfidf_vectorizer.pkl, yelp_xgb_model.json")
except Exception as e:
    print("Save skipped:", e)

In [ ]:
# 9) Quick inference helper (prints star names)
def predict_stars(texts):
    texts_clean = clean_with_gensim(texts)
    Xv = tfidf.transform(texts_clean)
    yp = clf.predict(Xv)
    return [(t, star_names[int(y)]) for t, y in zip(texts, yp)]

demo = ["Absolutely loved the food and service!",
        "Long wait, cold food. Not going back.",
        "Pretty average overall."]
print("\nDemo predictions:")
for t, lbl in predict_stars(demo):
    snip = (t[:80] + "…") if len(t) > 80 else t
    print(f" - {lbl} :: {snip}")

In [ ]:
from datasets import load_dataset
import pandas as pd

# -------------------------------
# 1) Load Yelp/yelp_review_full
#    (labels are 0..4 → 1..5 stars)
# -------------------------------
ds = load_dataset("Yelp/yelp_review_full")

# Use Python lists to avoid a huge pandas copy; keeps RAM saner
texts = ds["train"]["text"]
labels = np.array(ds["train"]["label"], dtype=np.int64)

# Quick sanity prints
unique, counts = np.unique(labels, return_counts=True)
print("Raw train distribution {label: count}:", dict(zip(unique, counts)))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

yelp_review_full/train-00000-of-00001.pa(…):   0%|          | 0.00/299M [00:00<?, ?B/s]

yelp_review_full/test-00000-of-00001.par(…):   0%|          | 0.00/23.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/650000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Raw train distribution {label: count}: {np.int64(0): np.int64(130000), np.int64(1): np.int64(130000), np.int64(2): np.int64(130000), np.int64(3): np.int64(130000), np.int64(4): np.int64(130000)}


In [ ]:
# df = ds["train"].to_pandas()[["text", "label"]]

# # Map 0–4 to star ratings (1–5)
# label_names = {0: "1 star (very negative)",
#                1: "2 stars (negative)",
#                2: "3 stars (neutral)",
#                3: "4 stars (positive)",
#                4: "5 stars (very positive)"}

# df["label_name"] = df["label"].map(label_names)

# print(df.sample(3)[["label", "label_name", "text"]])

#         label              label_name  \
# 519094      0  1 star (very negative)
# 168136      3      4 stars (positive)
# 358551      2       3 stars (neutral)

#                                                      text
# 519094  Wow. I wish I could give this place 0 stars, b...
# 168136  Our 2nd time here. We wanted to sit in the pat...
# 358551  The Paper Gallery is an amiable card shop, wit...

In [ ]:
# Use Python lists to avoid a huge pandas copy; keeps RAM saner
texts = ds["train"]["text"]
labels = np.array(ds["train"]["label"], dtype=np.int64)

In [ ]:
# Basic cleaning: drop NA and overly short rows
df = df.dropna(subset=["text", "label"]).reset_index(drop=True)
df = df[df["text"].str.strip().str.len() > 3].reset_index(drop=True)

print("Class counts (raw):")
print(df["label"].value_counts().sort_index())

In [ ]:
# -------------------------------
# 1) Encode labels
# -------------------------------
le = LabelEncoder()
y = le.fit_transform(df["label"].astype(str))  # cast to str to be safe across sources
X = df["text"].astype(str).values


In [ ]:
# -------------------------------
# 2) Stratified split (80/10/10)
# -------------------------------
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)

print("\nClass counts (train):", np.bincount(y_train))
print("Class counts (val):  ", np.bincount(y_val))
print("Class counts (test): ", np.bincount(y_test))

In [ ]:
# -------------------------------
# 3) TF-IDF (memory-conscious)
#    - float32 to halve memory
#    - prune rare/common tokens for large corpora
#    - ngrams (1,2) to capture short phrases ("not good")
# -------------------------------
tfidf = TfidfVectorizer(
    stop_words="english",
    lowercase=True,
    dtype=np.float32,       # big memory win
    max_features=200_000,   # adjust to your RAM (50k–300k typical)
    ngram_range=(1, 2),
    min_df=5,               # tighten for very large datasets (5–10 is common)
    max_df=0.95,
    sublinear_tf=True,
    norm="l2",
)

Xtr = tfidf.fit_transform(X_train)
Xva = tfidf.transform(X_val)
Xte = tfidf.transform(X_test)

assert sp.issparse(Xtr) and Xtr.dtype == np.float32

In [ ]:
# -------------------------------
# 4) Imbalance handling via per-example weights (multi-class)
# -------------------------------
classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight_map = {c: float(weights[i]) for i, c in enumerate(classes)}
sample_weights = np.vectorize(class_weight_map.get)(y_train).astype(np.float32)

In [ ]:
# -------------------------------
# 5) XGBoost config for large sparse text
#    - Use "gpu_hist" if you have a compatible NVIDIA GPU
#    - Early stopping to cut training time and overfitting
# -------------------------------
tree_method = "gpu_hist" if os.environ.get("XGB_USE_GPU", "0") == "1" else "hist"

clf = XGBClassifier(
    objective="multi:softprob",
    num_class=len(classes),
    n_estimators=2000,        # high cap; early stopping will pick best iteration
    learning_rate=0.05,
    max_depth=8,              # try 6–10 for text
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    reg_alpha=0.0,
    tree_method=tree_method,  # "hist" (CPU) or "gpu_hist"
    eval_metric="mlogloss",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

clf.fit(
    Xtr, y_train,
    sample_weight=sample_weights,
    eval_set=[(Xva, y_val)],
    early_stopping_rounds=50,
    verbose=100
)

In [ ]:
# -------------------------------
# 6) Evaluation: accuracy + macro/weighted F1 + confusion matrix
# -------------------------------
y_pred = clf.predict(Xte)

print("\nAccuracy:", f"{accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_, digits=4))

cm = confusion_matrix(y_test, y_pred, normalize="true")
cm_df = pd.DataFrame(cm, index=le.classes_, columns=le.classes_)
print("\nNormalized Confusion Matrix (rows=true):")
print(cm_df.round(3))

In [ ]:
# -------------------------------
# 7) Persist components (reproducible inference)
# -------------------------------
joblib.dump(tfidf, "tfidf_vectorizer.pkl", compress=3)
joblib.dump(le, "label_encoder.pkl", compress=3)
clf.save_model("xgb_model.json")           # portable XGBoost format
# Optional: joblib.dump(clf, "xgb_clf.pkl", compress=3)

print("\nSaved: tfidf_vectorizer.pkl, label_encoder.pkl, xgb_model.json")

In [ ]:
# -------------------------------
# 8) Quick inference helper
# -------------------------------
def predict_texts(texts):
    Xv = tfidf.transform(texts)
    yp = clf.predict(Xv)
    return list(zip(texts, le.inverse_transform(yp)))

demo = ["Absolutely loved it!", "Terrible quality. Not recommended."]
print("\nDemo predictions:", predict_texts(demo))

In [ ]:
# # tfidf_xgb_imbalanced.py
# # Robust TF-IDF + XGBoost pipeline for large, imbalanced corpora

# import os, gc, numpy as np, pandas as pd
# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import LabelEncoder
# from sklearn.utils.class_weight import compute_class_weight
# from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
# from sklearn.feature_extraction.text import TfidfVectorizer
# from xgboost import XGBClassifier
# import joblib
# import scipy.sparse as sp

# RANDOM_STATE = 42

# # -------------------------------
# # 0) Choose ONE loader
# #    A) Hugging Face dataset example (Amazon reviews, 5-class, realistically imbalanced)
# #    B) Local CSV: columns ["text","label"]
# # -------------------------------

# USE_HF = True

# if USE_HF:
#     # pip install datasets
#     from datasets import load_dataset

#     # Example 1: Multilingual Amazon Reviews (pick one language, e.g., English "en")
#     # NOTE: Amazon reviews commonly skew toward 5-star; good for imbalance testing
#     # Alt: "McAuley-Lab/Amazon-Reviews-2023" (very large); or "google/civil_comments" (binary, toxic minority)
#     ds = load_dataset("mteb/amazon_reviews_multi", name="en")
#     # Keep only text + star label
#     df = pd.DataFrame({
#         "text": ds["train"]["review_body"],
#         "label": ds["train"]["stars"],  # 1..5
#     })
# else:
#     # Local CSV (replace path). Must have columns: text,label
#     CSV_PATH = "your_file.csv"
#     df = pd.read_csv(CSV_PATH)


# from datasets import load_dataset
# ds = load_dataset("google/civil_comments")
# import pandas as pd
# df = pd.DataFrame({"label": ds["train"]["toxic"]})
# print(df["label"].value_counts(normalize=True))  # expect strong skew (few toxics)

# # Basic cleaning: drop NA and overly short rows
# df = df.dropna(subset=["text", "label"]).reset_index(drop=True)
# df = df[df["text"].str.strip().str.len() > 3].reset_index(drop=True)

# print("Class counts (raw):")
# print(df["label"].value_counts().sort_index())

# # -------------------------------
# # 1) Encode labels
# # -------------------------------
# le = LabelEncoder()
# y = le.fit_transform(df["label"].astype(str))  # cast to str to be safe across sources
# X = df["text"].astype(str).values

# # -------------------------------
# # 2) Stratified split (80/10/10)
# # -------------------------------
# X_train, X_temp, y_train, y_temp = train_test_split(
#     X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
# )
# X_val, X_test, y_val, y_test = train_test_split(
#     X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
# )

# print("\nClass counts (train):", np.bincount(y_train))
# print("Class counts (val):  ", np.bincount(y_val))
# print("Class counts (test): ", np.bincount(y_test))

# # -------------------------------
# # 3) TF-IDF (memory-conscious)
# #    - float32 to halve memory
# #    - prune rare/common tokens for large corpora
# #    - ngrams (1,2) to capture short phrases ("not good")
# # -------------------------------
# tfidf = TfidfVectorizer(
#     stop_words="english",
#     lowercase=True,
#     dtype=np.float32,       # big memory win
#     max_features=200_000,   # adjust to your RAM (50k–300k typical)
#     ngram_range=(1, 2),
#     min_df=5,               # tighten for very large datasets (5–10 is common)
#     max_df=0.95,
#     sublinear_tf=True,
#     norm="l2",
# )

# Xtr = tfidf.fit_transform(X_train)
# Xva = tfidf.transform(X_val)
# Xte = tfidf.transform(X_test)

# assert sp.issparse(Xtr) and Xtr.dtype == np.float32

# # -------------------------------
# # 4) Imbalance handling via per-example weights (multi-class)
# # -------------------------------
# classes = np.unique(y_train)
# weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
# class_weight_map = {c: float(weights[i]) for i, c in enumerate(classes)}
# sample_weights = np.vectorize(class_weight_map.get)(y_train).astype(np.float32)

# # -------------------------------
# # 5) XGBoost config for large sparse text
# #    - Use "gpu_hist" if you have a compatible NVIDIA GPU
# #    - Early stopping to cut training time and overfitting
# # -------------------------------
# tree_method = "gpu_hist" if os.environ.get("XGB_USE_GPU", "0") == "1" else "hist"

# clf = XGBClassifier(
#     objective="multi:softprob",
#     num_class=len(classes),
#     n_estimators=2000,        # high cap; early stopping will pick best iteration
#     learning_rate=0.05,
#     max_depth=8,              # try 6–10 for text
#     subsample=0.8,
#     colsample_bytree=0.8,
#     reg_lambda=1.0,
#     reg_alpha=0.0,
#     tree_method=tree_method,  # "hist" (CPU) or "gpu_hist"
#     eval_metric="mlogloss",
#     n_jobs=-1,
#     random_state=RANDOM_STATE,
# )

# clf.fit(
#     Xtr, y_train,
#     sample_weight=sample_weights,
#     eval_set=[(Xva, y_val)],
#     early_stopping_rounds=50,
#     verbose=100
# )

# # -------------------------------
# # 6) Evaluation: accuracy + macro/weighted F1 + confusion matrix
# # -------------------------------
# y_pred = clf.predict(Xte)

# print("\nAccuracy:", f"{accuracy_score(y_test, y_pred):.4f}")
# print("\nClassification Report:")
# print(classification_report(y_test, y_pred, target_names=le.classes_, digits=4))

# cm = confusion_matrix(y_test, y_pred, normalize="true")
# cm_df = pd.DataFrame(cm, index=le.classes_, columns=le.classes_)
# print("\nNormalized Confusion Matrix (rows=true):")
# print(cm_df.round(3))

# # -------------------------------
# # 7) Persist components (reproducible inference)
# # -------------------------------
# joblib.dump(tfidf, "tfidf_vectorizer.pkl", compress=3)
# joblib.dump(le, "label_encoder.pkl", compress=3)
# clf.save_model("xgb_model.json")           # portable XGBoost format
# # Optional: joblib.dump(clf, "xgb_clf.pkl", compress=3)

# print("\nSaved: tfidf_vectorizer.pkl, label_encoder.pkl, xgb_model.json")

# # -------------------------------
# # 8) Quick inference helper
# # -------------------------------
# def predict_texts(texts):
#     Xv = tfidf.transform(texts)
#     yp = clf.predict(Xv)
#     return list(zip(texts, le.inverse_transform(yp)))

# demo = ["Absolutely loved it!", "Terrible quality. Not recommended."]
# print("\nDemo predictions:", predict_texts(demo))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/47.0 [00:00<?, ?B/s]

amazon_reviews_multi.py: 0.00B [00:00, ?B/s]

RuntimeError: Dataset scripts are no longer supported, but found amazon_reviews_multi.py